<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content, on one day, for one client (content-day grain), from fact_content_daily_performance. Developing on a mid-panel month (month=2026-03); holding out the final month (June 2026) as a sealed test window, since the last month is the natural outcome window for any past→future label.

In [14]:
cols = con.sql(f"DESCRIBE SELECT * FROM {REL}").df()
for c in cols['column_name']:
    print(c)

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [15]:
con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {REL}
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



One row = one piece of content (content_hash_id), for one client (client_hash_id), on one day (report_date), from fact_content_daily_performance. Developing on a mid-panel month (month=2026-03); holding out the final month (June 2026) as a sealed test window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (knowable at report time): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, sessions_organic, sessions_ai (and its breakdown: ai_claude, ai_chatgpt, ai_perplexity). Label / proxy: no ready-made label exists in this table — the proxy will be an observed trend comparing a content item's gsc_clicks in this month against its own prior month, computed directly from the data. Context (grouping/joining only): content_hash_id, client_hash_id, report_date, month. Excluded: client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — these are data-availability flags, not performance signals, and including them risks confusing "we don't have data" with "the content performed poorly."

In [16]:
con.sql(f"""
    SELECT gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions,
           ga4_engaged_sessions, sessions_organic, sessions_ai,
           ai_claude, ai_chatgpt, ai_perplexity
    FROM {REL}
    LIMIT 5
""").show()

┌─────────────────┬────────────┬───────────────────┬──────────────┬──────────────────────┬──────────────────┬─────────────┬───────────┬────────────┬───────────────┐
│ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ ga4_sessions │ ga4_engaged_sessions │ sessions_organic │ sessions_ai │ ai_claude │ ai_chatgpt │ ai_perplexity │
│      int64      │   int64    │      double       │    int64     │        int64         │      int64       │    int64    │   int64   │   int64    │     int64     │
├─────────────────┼────────────┼───────────────────┼──────────────┼──────────────────────┼──────────────────┼─────────────┼───────────┼────────────┼───────────────┤
│              20 │          0 │              3.35 │         NULL │                 NULL │             NULL │        NULL │      NULL │       NULL │          NULL │
│               1 │          0 │               0.0 │         NULL │                 NULL │             NULL │        NULL │      NULL │       NULL │          NULL │
│         

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verifying three claims: the grain holds (no duplicate content-client-day rows), missingness on key features isn't random (checked against the availability flags), and the date window matches the stated month.

In [17]:
# 1. Grain check — should return ZERO rows if one row = one content-client-day
con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) c
    FROM {REL}
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").show()

# 2. Missingness — compare missing clicks to GSC availability flag
con.sql(f"""
    SELECT
        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_clicks,
        AVG(CASE WHEN gsc_data_available = FALSE THEN 1.0 ELSE 0 END) AS pct_gsc_unavailable
    FROM {REL}
""").show()

# 3. Window check
con.sql(f"SELECT MIN(report_date), MAX(report_date), COUNT(DISTINCT client_hash_id) AS n_clients FROM {REL}").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────┬─────────────┬───────┐
│ content_hash_id │ client_hash_id │ report_date │   c   │
│     varchar     │    varchar     │    date     │ int64 │
├─────────────────┴────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

┌────────────────────┬─────────────────────┐
│ pct_missing_clicks │ pct_gsc_unavailable │
│       double       │       double        │
├────────────────────┼─────────────────────┤
│                0.0 │  0.6330736407035681 │
└────────────────────┴─────────────────────┘

┌──────────────────┬──────────────────┬───────────┐
│ min(report_date) │ max(report_date) │ n_clients │
│       date       │       date       │   int64   │
├──────────────────┼──────────────────┼───────────┤
│ 2026-03-01       │ 2026-03-31       │        55 │
└──────────────────┴──────────────────┴───────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

A major limitation: gsc_clicks is never NULL, but gsc_data_available is FALSE for 63.3% of rows — meaning when GSC data isn't actually available, clicks default to 0 rather than staying blank. This risks confusing "no data was collected" with "the content genuinely got zero clicks." Any feature or proxy built from gsc_clicks must first filter to gsc_data_available = TRUE, or it will silently undercount performance for the majority of rows. Separately, this warehouse is an unbalanced panel — client history depth differs by client, so this one month cannot represent clients whose data coverage started later.

In [18]:
# Confirm: among rows where GSC IS available, is clicks still ever legitimately 0? (sanity check)
con.sql(f"""
    SELECT
        gsc_data_available,
        AVG(gsc_clicks) AS avg_clicks,
        COUNT(*) AS n_rows
    FROM {REL}
    GROUP BY gsc_data_available
""").show()

┌────────────────────┬─────────────────────┬─────────┐
│ gsc_data_available │     avg_clicks      │ n_rows  │
│      boolean       │       double        │  int64  │
├────────────────────┼─────────────────────┼─────────┤
│ false              │                 0.0 │ 6230317 │
│ true               │ 0.22758740436674982 │ 3611061 │
└────────────────────┴─────────────────────┴─────────┘



Confirmed: rows with gsc_data_available = false have an average of exactly 0.0 clicks across all 6.2M rows — a mathematically perfect zero, which is not how real click data behaves. Rows with gsc_data_available = true average 0.23 clicks, a realistic distribution. This confirms the zero is a missing-data artifact, not a true measurement.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.